# AlphaZero 렌주 15×15 — Colab GPU 학습

**실행 순서**: 셀을 위에서 아래로 순서대로 실행하세요.  
**전제**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 연결

| 셀 | 내용 |
|---|---|
| 1 | Google Drive 마운트 (체크포인트 영구 저장) |
| 2 | 저장소 클론 / 업데이트 |
| 3 | GPU 상태 확인 |
| 4 | GPU 스모크 테스트 (3iter → 본 학습 규모 결정) |
| 5 | 본 학습 (스모크 결과 보고 값 조정 후 실행) |
| 6 | 재개 (세션 끊긴 후 이어서 학습) |

In [ ]:
# ── 셀 1: Google Drive 마운트 ────────────────────────────────────────────
# 체크포인트를 Drive에 저장해 세션 끊겨도 보존됩니다.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_CKPT = '/content/drive/MyDrive/rl_omok/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
print(f'Drive 마운트 완료. 체크포인트: {DRIVE_CKPT}')
print('파일 목록:', os.listdir(DRIVE_CKPT) if os.listdir(DRIVE_CKPT) else '(비어 있음)')

In [ ]:
# ── 셀 2: 저장소 클론 / 업데이트 ────────────────────────────────────────
# 처음 실행: 클론
# 재실행 (세션 재시작 후): 이미 있으면 pull
import os

REPO_URL = 'https://github.com/namu627/RL-omok'
REPO_DIR = '/content/RL-omok'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3
print('현재 디렉터리:', os.getcwd())

In [ ]:
# ── 셀 3: GPU / 환경 확인 ───────────────────────────────────────────────
import torch, sys

print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {gb:.1f} GB')
else:
    print('⚠️  GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU 선택')

In [ ]:
# ── 셀 4: GPU 스모크 테스트 ─────────────────────────────────────────────
# 3iter 실행 → iter당 시간 측정 → 본 학습 규모 결정
#
# 출력 마지막에 본 학습 예상 시간 표가 나옵니다.
# 그 표를 보고 셀 5의 FULL_* 값을 조정하세요.

!python scripts/train_renju_colab.py smoke

In [ ]:
# ── 셀 5: 본 학습 ────────────────────────────────────────────────────────
# 스모크 결과를 보고 아래 값을 조정하세요.
# 수정 후 이 셀만 재실행하면 됩니다.
#
# train_renju_colab.py 상단의 FULL_* 상수를 직접 수정하거나,
# 아래처럼 환경변수로 덮어씁니다.

# --- 값 조정 ---
import subprocess, os

# train_renju_colab.py 의 상수를 sed로 덮어쓰기 (선택적)
# 예시: n_sim=200, 25게임/iter, 100iter
# !sed -i 's/^FULL_N_SIM.*/FULL_N_SIM      = 200/' scripts/train_renju_colab.py
# !sed -i 's/^FULL_SP_GAMES.*/FULL_SP_GAMES   = 25/'  scripts/train_renju_colab.py
# !sed -i 's/^FULL_N_ITER.*/FULL_N_ITER     = 100/'   scripts/train_renju_colab.py

# --- 실행 ---
!python scripts/train_renju_colab.py train

In [ ]:
# ── 셀 6: 재개 (세션 끊긴 후) ───────────────────────────────────────────
# 세션이 끊기면:
#   1. 런타임 재연결
#   2. 셀 1 (Drive 마운트) 실행
#   3. 셀 2 (저장소 클론) 실행
#   4. 이 셀 실행
#
# --resume 플래그: ckpt_dir에서 가장 최근 iter 체크포인트를 자동으로 로드합니다.

import os
DRIVE_CKPT = '/content/drive/MyDrive/rl_omok/checkpoints'
ckpts = sorted([f for f in os.listdir(DRIVE_CKPT) if f.endswith('.pt')])
print('저장된 체크포인트:', ckpts)

!python scripts/train_renju_colab.py train --resume

In [ ]:
# ── 셀 7: 결과 확인 ──────────────────────────────────────────────────────
import json, os
from IPython.display import Image

DRIVE_DIR = '/content/drive/MyDrive/rl_omok'

# JSON 결과
json_path = os.path.join(DRIVE_DIR, 'checkpoints', 'full_results.json')
if os.path.exists(json_path):
    with open(json_path) as f:
        r = json.load(f)
    print(f"총 학습 시간 : {r['total_sec']/3600:.2f}h")
    print(f"Device       : {r['device']}")
    print(f"설정         : n_sim={r['n_sim']}, {r['sp_games']}games/iter, {r['n_iter']}iter")
    print(f"승률 이력    : {[round(w,3) for w in r['win_rates']]}")

# 승률 곡선 이미지
plot_path = os.path.join(DRIVE_DIR, 'az_renju_winrate.png')
if os.path.exists(plot_path):
    display(Image(plot_path))